# 01 Finance Data Engineering — Ingestion Pipeline

## Project Context

This notebook implements Phase 0 of the Finance Data Engineering Stack — a portfolio project demonstrating a production-style market data ingestion and transformation pipeline in Python.

**Disclaimer:** This is a portfolio data engineering project. Nothing here constitutes investment advice or a production trading system.

## Ingestion Objective

1. Download adjusted close prices for a defined asset universe via yfinance.
2. Save raw OHLCV data for auditability.
3. Extract and clean the adjusted close price panel.
4. Compute daily simple returns.
5. Build SQL-ready dimension and fact tables.
6. Log the ingestion run metadata.
7. Save all outputs to `data/processed/` and `outputs/`.

## Asset Universe

| Ticker | Sector |
|---|---|
| AAPL | Technology |
| MSFT | Technology |
| NVDA | Technology |
| JPM  | Financials |
| PG   | Consumer Staples |
| KO   | Consumer Staples |
| XOM  | Energy |
| JNJ  | Health Care |
| SPY  | ETF (S&P 500) |

In [1]:
import os
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

# Resolve project root from notebook CWD (nbconvert sets CWD = notebook dir)
NOTEBOOK_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(5):
    if (PROJECT_ROOT / "src").is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RAW_DATA_DIR, PROCESSED_DATA_DIR,
    LOGS_DIR, SUMMARIES_DIR,
    ASSET_UNIVERSE, DEFAULT_START_DATE,
)
from src.ingestion import (
    download_market_data, extract_adjusted_close,
    save_raw_market_data, save_processed_market_data,
    create_ingestion_log,
)
from src.transforms import (
    calculate_returns, create_asset_dimension,
    create_price_fact_table, create_returns_fact_table,
)

TICKERS    = ASSET_UNIVERSE
START_DATE = DEFAULT_START_DATE
RUN_TS     = datetime.now(tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

print(f"Project root : {PROJECT_ROOT}")
print(f"Tickers      : {TICKERS}")
print(f"Start date   : {START_DATE}")
print(f"Run timestamp: {RUN_TS}")

Project root : /Users/rovs/Documents/New project 2/projects/finance-data-engineering-stack
Tickers      : ['AAPL', 'MSFT', 'JPM', 'PG', 'XOM', 'JNJ', 'KO', 'NVDA', 'SPY']
Start date   : 2020-01-01
Run timestamp: 2026-04-30 12:54:00


## Download Market Data (yfinance)

In [2]:
raw_data = download_market_data(TICKERS, start_date=START_DATE)

print(f"Raw data shape : {raw_data.shape}")
print(f"Date range     : {raw_data.index.min().date()} to {raw_data.index.max().date()}")
print(f"Columns (first 10): {list(raw_data.columns[:10])}")

Raw data shape : (1589, 54)
Date range     : 2020-01-02 to 2026-04-29
Columns (first 10): [('Adj Close', 'AAPL'), ('Adj Close', 'JNJ'), ('Adj Close', 'JPM'), ('Adj Close', 'KO'), ('Adj Close', 'MSFT'), ('Adj Close', 'NVDA'), ('Adj Close', 'PG'), ('Adj Close', 'SPY'), ('Adj Close', 'XOM'), ('Close', 'AAPL')]


## Save Raw Market Data

In [3]:
raw_path = RAW_DATA_DIR / "market_prices_raw.csv"
save_raw_market_data(raw_data, raw_path)
print(f"Raw data saved: {raw_path.name}  ({raw_path.stat().st_size:,} bytes)")

Raw data saved: market_prices_raw.csv  (1,420,946 bytes)


## Extract Adjusted Close Prices

In [4]:
prices = extract_adjusted_close(raw_data)

print(f"Prices shape : {prices.shape}")
print(f"Tickers      : {list(prices.columns)}")
print(f"Date range   : {prices.index.min().date()} to {prices.index.max().date()}")
print(f"\nLatest prices:")
print(prices.tail(3).round(2).to_string())

Prices shape : (1589, 9)
Tickers      : ['AAPL', 'JNJ', 'JPM', 'KO', 'MSFT', 'NVDA', 'PG', 'SPY', 'XOM']
Date range   : 2020-01-02 to 2026-04-29

Latest prices:
ticker        AAPL     JNJ     JPM     KO    MSFT    NVDA      PG     SPY     XOM
date                                                                             
2026-04-27  267.61  225.34  311.63  75.44  424.82  216.61  148.40  715.17  148.19
2026-04-28  270.71  227.79  311.45  78.35  429.25  213.17  149.17  711.69  150.56
2026-04-29  270.17  227.35  309.25  78.87  424.46  209.25  146.46  711.58  154.67


## Save Processed Adjusted Close Prices

In [5]:
prices_path = PROCESSED_DATA_DIR / "adjusted_close_prices.csv"
save_processed_market_data(prices, prices_path)
print(f"Saved: {prices_path.name}  ({prices_path.stat().st_size:,} bytes)")

Saved: adjusted_close_prices.csv  (279,092 bytes)


## Compute Daily Returns

In [6]:
returns = calculate_returns(prices)

returns_path = PROCESSED_DATA_DIR / "daily_returns.csv"
save_processed_market_data(returns, returns_path)

print(f"Returns shape: {returns.shape}")
print(f"Saved       : {returns_path.name}  ({returns_path.stat().st_size:,} bytes)")
print(f"\nSample returns (last 3 rows):")
print(returns.tail(3).round(5).to_string())

Returns shape: (1589, 9)
Saved       : daily_returns.csv  (323,929 bytes)

Sample returns (last 3 rows):
ticker         AAPL      JNJ      JPM       KO     MSFT     NVDA       PG      SPY      XOM
date                                                                                       
2026-04-27 -0.01273 -0.00949  0.01087 -0.01553  0.00047  0.04004  0.00148  0.00172 -0.00484
2026-04-28  0.01158  0.01087 -0.00058  0.03857  0.01043 -0.01588  0.00519 -0.00487  0.01599
2026-04-29 -0.00199 -0.00193 -0.00706  0.00664 -0.01116 -0.01839 -0.01817 -0.00015  0.02730


## Build Asset Dimension Table

In [7]:
dim_assets = create_asset_dimension(TICKERS)

dim_path = PROCESSED_DATA_DIR / "dim_assets.csv"
dim_assets.to_csv(dim_path, index=False)

print(f"dim_assets rows : {len(dim_assets)}")
print(f"Saved           : {dim_path.name}")
print()
print(dim_assets.to_string(index=False))

dim_assets rows : 9
Saved           : dim_assets.csv

 asset_id ticker       asset_type           loaded_at
        1   AAPL       Technology 2026-04-30 12:54:03
        2    JNJ      Health Care 2026-04-30 12:54:03
        3    JPM       Financials 2026-04-30 12:54:03
        4     KO Consumer Staples 2026-04-30 12:54:03
        5   MSFT       Technology 2026-04-30 12:54:03
        6   NVDA       Technology 2026-04-30 12:54:03
        7     PG Consumer Staples 2026-04-30 12:54:03
        8    SPY              ETF 2026-04-30 12:54:03
        9    XOM           Energy 2026-04-30 12:54:03


## Build Price Fact Table

In [8]:
fact_prices = create_price_fact_table(prices)

fp_path = PROCESSED_DATA_DIR / "fact_prices.csv"
fact_prices.to_csv(fp_path, index=False)

print(f"fact_prices rows: {len(fact_prices):,}")
print(f"Columns         : {list(fact_prices.columns)}")
print(f"Saved           : {fp_path.name}  ({fp_path.stat().st_size:,} bytes)")
print()
print(fact_prices.tail(5).to_string(index=False))

fact_prices rows: 14,301
Columns         : ['price_id', 'date', 'ticker', 'adj_close']
Saved           : fact_prices.csv  (552,406 bytes)

 price_id       date ticker  adj_close
    14297 2026-04-29   MSFT 424.459991
    14298 2026-04-29   NVDA 209.250000
    14299 2026-04-29     PG 146.460007
    14300 2026-04-29    SPY 711.580017
    14301 2026-04-29    XOM 154.669998


## Build Returns Fact Table

In [9]:
fact_returns = create_returns_fact_table(returns)

fr_path = PROCESSED_DATA_DIR / "fact_returns.csv"
fact_returns.to_csv(fr_path, index=False)

print(f"fact_returns rows: {len(fact_returns):,}")
print(f"Columns          : {list(fact_returns.columns)}")
print(f"Saved            : {fr_path.name}  ({fr_path.stat().st_size:,} bytes)")
print()
print(fact_returns.tail(5).to_string(index=False))

fact_returns rows: 14,292
Columns          : ['return_id', 'date', 'ticker', 'daily_return']
Saved            : fact_returns.csv  (597,048 bytes)

 return_id       date ticker  daily_return
     14288 2026-04-29   MSFT     -0.011159
     14289 2026-04-29   NVDA     -0.018389
     14290 2026-04-29     PG     -0.018167
     14291 2026-04-29    SPY     -0.000155
     14292 2026-04-29    XOM      0.027298


## Build Ingestion Log

In [10]:
log_records = []
for ticker in TICKERS:
    col_data = prices[ticker].dropna() if ticker in prices.columns else pd.Series(dtype=float)
    log_records.append({
        "ticker":          ticker,
        "start_date":      START_DATE,
        "end_date":        str(prices.index.max().date()),
        "rows_downloaded": len(col_data),
        "missing_count":   int(prices[ticker].isnull().sum()) if ticker in prices.columns else -1,
        "latest_price":    round(float(col_data.iloc[-1]), 4) if not col_data.empty else None,
        "status":          "ok" if not col_data.empty else "no_data",
        "run_timestamp":   RUN_TS,
    })

log_path = LOGS_DIR / "ingestion_log.csv"
log_df = create_ingestion_log(log_records, log_path)

print(f"Ingestion log saved: {log_path.name}")
print(log_df.to_string(index=False))

Ingestion log saved: ingestion_log.csv
ticker start_date   end_date  rows_downloaded  missing_count  latest_price status       run_timestamp
  AAPL 2020-01-01 2026-04-29             1589              0        270.17     ok 2026-04-30 12:54:00
  MSFT 2020-01-01 2026-04-29             1589              0        424.46     ok 2026-04-30 12:54:00
   JPM 2020-01-01 2026-04-29             1589              0        309.25     ok 2026-04-30 12:54:00
    PG 2020-01-01 2026-04-29             1589              0        146.46     ok 2026-04-30 12:54:00
   XOM 2020-01-01 2026-04-29             1589              0        154.67     ok 2026-04-30 12:54:00
   JNJ 2020-01-01 2026-04-29             1589              0        227.35     ok 2026-04-30 12:54:00
    KO 2020-01-01 2026-04-29             1589              0         78.87     ok 2026-04-30 12:54:00
  NVDA 2020-01-01 2026-04-29             1589              0        209.25     ok 2026-04-30 12:54:00
   SPY 2020-01-01 2026-04-29             15

## Data Summary

In [11]:
# Descriptive stats on adjusted close prices
print("=== Adjusted Close Price Summary ===")
print(prices.describe().round(2).to_string())

print()
print("=== Daily Return Summary (%) ===")
print((returns * 100).describe().round(4).to_string())

# Missing value check
print()
print("=== Missing Value Counts ===")
print(prices.isnull().sum().to_string())

=== Adjusted Close Price Summary ===
ticker     AAPL      JNJ      JPM       KO     MSFT     NVDA       PG      SPY      XOM
count   1589.00  1589.00  1589.00  1589.00  1589.00  1589.00  1589.00  1589.00  1589.00
mean     169.70   151.89   166.03    55.58   320.45    64.13   136.38   452.01    83.13
std       52.87    24.59    69.63     9.92   100.99    62.07    18.22   118.60    32.56
min       54.21    93.97    66.76    31.30   128.64     4.89    83.41   204.94    23.99
25%      133.32   140.57   117.06    48.21   237.81    15.13   122.73   371.26    50.96
50%      167.26   148.43   138.76    55.56   304.08    27.94   138.48   420.38    95.26
75%      210.15   156.02   208.83    61.20   408.49   118.05   150.43   548.04   106.81
max      285.92   248.56   332.91    81.00   539.83   216.61   172.54   715.17   171.47

=== Daily Return Summary (%) ===
ticker       AAPL        JNJ        JPM         KO       MSFT       NVDA         PG        SPY        XOM
count   1588.0000  1588.0000  1

In [12]:
# Save ingestion summary
summary_records = [{
    "run_timestamp":      RUN_TS,
    "tickers_requested":  len(TICKERS),
    "tickers_ingested":   int((log_df["status"] == "ok").sum()),
    "date_range_start":   START_DATE,
    "date_range_end":     str(prices.index.max().date()),
    "trading_days":       len(prices),
    "total_price_rows":   len(fact_prices),
    "total_return_rows":  len(fact_returns),
    "raw_file":           str(raw_path.name),
    "prices_file":        str(prices_path.name),
    "returns_file":       str(returns_path.name),
    "dim_assets_file":    str(dim_path.name),
    "fact_prices_file":   str(fp_path.name),
    "fact_returns_file":  str(fr_path.name),
}]

summary_df = pd.DataFrame(summary_records)
summary_path = SUMMARIES_DIR / "ingestion_summary.csv"
summary_df.to_csv(summary_path, index=False)

print(f"Summary saved: {summary_path.name}")
print(summary_df.T.to_string(header=False))

Summary saved: ingestion_summary.csv
run_timestamp            2026-04-30 12:54:00
tickers_requested                          9
tickers_ingested                           9
date_range_start                  2020-01-01
date_range_end                    2026-04-29
trading_days                            1589
total_price_rows                       14301
total_return_rows                      14292
raw_file               market_prices_raw.csv
prices_file        adjusted_close_prices.csv
returns_file               daily_returns.csv
dim_assets_file               dim_assets.csv
fact_prices_file             fact_prices.csv
fact_returns_file           fact_returns.csv


## Output Verification

In [13]:
output_files = [
    raw_path,
    prices_path,
    returns_path,
    dim_path,
    fp_path,
    fr_path,
    log_path,
    summary_path,
]

print("Phase 0 output verification:")
print("-" * 65)
all_ok = True
for p in output_files:
    exists = p.exists()
    size   = p.stat().st_size if exists else 0
    status = "OK" if (exists and size > 0) else "MISSING"
    if status != "OK":
        all_ok = False
    rel = p.relative_to(PROJECT_ROOT)
    print(f"  {status:<8} {size:>10,} bytes  {rel}")
print("-" * 65)
print("All outputs verified." if all_ok else "WARNING: one or more outputs missing.")

Phase 0 output verification:
-----------------------------------------------------------------
  OK        1,420,946 bytes  data/raw/market_prices_raw.csv
  OK          279,092 bytes  data/processed/adjusted_close_prices.csv
  OK          323,929 bytes  data/processed/daily_returns.csv
  OK              373 bytes  data/processed/dim_assets.csv
  OK          552,406 bytes  data/processed/fact_prices.csv
  OK          597,048 bytes  data/processed/fact_returns.csv
  OK              658 bytes  outputs/logs/ingestion_log.csv
  OK              391 bytes  outputs/summaries/ingestion_summary.csv
-----------------------------------------------------------------
All outputs verified.


## Limitations

- **yfinance dependency:** Data depends on Yahoo Finance availability. If the service is unavailable or rate-limits the request, ingestion fails. A CSV fallback is a Phase 1 improvement.
- **No validation yet:** Missing values, stale prices, and return outliers are not checked here. Phase 1 implements the full validation layer.
- **No warehouse yet:** Outputs are CSV files. DuckDB population and SQL querying are Phase 1 deliverables.
- **Batch ingestion only:** The pipeline downloads the full history on each run. Incremental ingestion is a future improvement.
- **Adjusted close only:** OHLCV data is saved in raw form but only adjusted close is used in processed tables. Volume and open/high/low are Phase 1 candidates.

## Next Steps for Phase 1

1. Implement data quality checks in `src/validation.py` — missing values, price staleness, return outliers.
2. Populate DuckDB warehouse from the fact/dim CSVs.
3. Run `sql/sample_queries.sql` against the warehouse.
4. Build `notebooks/02_data_quality_and_warehouse.ipynb`.
5. Generate `reports/data_quality_report.md`.